In [ ]:
# ============================================================
# Section 1: Imports and Photos Library paths
# ============================================================

import osxphotos

from photos_inventory import *


USE_INVENTORY_CACHE = True


PHOTOS_LIBRARY_PATHS = {
    "backup_20250317": "/Volumes/PRO-G40--20250315/Backup -- PRO-G40--20250315/Photos Library-iCloud-20250317（iCloud 20250325崩潰前最後的備份）.photoslibrary",
    "snapshot_20260603": "/Volumes/PRO-G40--20260519/Photos Library-Snapshot--20260603181919/Photos Library.photoslibrary",
    "test": "/Users/huohsien/Pictures/test.photoslibrary"
}


In [ ]:
# ============================================================
# Section 2: Load or build inventories
# ============================================================

if USE_INVENTORY_CACHE:
    print("=" * 80)
    print("Load inventory cache: backup_20250317")
    print("=" * 80)

    inventory_backup = load_inventory_cache("backup_20250317")

    print()
    print("=" * 80)
    print("Load inventory cache: snapshot_20260603")
    print("=" * 80)

    inventory_snapshot = load_inventory_cache("snapshot_20260603")

else:
    print("=" * 80)
    print("Build inventory: backup_20250317")
    print("=" * 80)

    osx_assets = osxphotos.PhotosDB(
        PHOTOS_LIBRARY_PATHS["backup_20250317"]
    ).photos()

    print("backup osx asset count:", len(osx_assets))

    inventory_backup = build_inventory(osx_assets)

    print()
    print("Backup inventory summary")
    print("------------------------")
    print_inventory_summary(inventory_backup)

    save_inventory_cache(inventory_backup, "backup_20250317")

    print()
    print("=" * 80)
    print("Build inventory: snapshot_20260603")
    print("=" * 80)

    osx_assets_snapshot = osxphotos.PhotosDB(
        PHOTOS_LIBRARY_PATHS["snapshot_20260603"]
    ).photos()

    print("snapshot osx asset count:", len(osx_assets_snapshot))

    inventory_snapshot = build_inventory(osx_assets_snapshot)

    print()
    print("Snapshot inventory summary")
    print("--------------------------")
    print_inventory_summary(inventory_snapshot)

    save_inventory_cache(inventory_snapshot, "snapshot_20260603")


In [ ]:
# ============================================================
# Test 2: Inventory comparison helpers
# ============================================================

from enum import Enum


class ChangeType(str, Enum):
    # Asset existence
    ASSET_MISSING_FROM_SNAPSHOT = "ASSET_MISSING_FROM_SNAPSHOT"
    ASSET_NEW_IN_SNAPSHOT = "ASSET_NEW_IN_SNAPSHOT"

    # Asset metadata fields
    ASSET_FIELD_CHANGED_DESCRIPTION = "ASSET_FIELD_CHANGED__description"
    ASSET_FIELD_CHANGED_KEYWORDS = "ASSET_FIELD_CHANGED__keywords"
    ASSET_FIELD_CHANGED_FAVORITE = "ASSET_FIELD_CHANGED__favorite"
    ASSET_FIELD_CHANGED_HIDDEN = "ASSET_FIELD_CHANGED__hidden"
    ASSET_FIELD_CHANGED_DATE = "ASSET_FIELD_CHANGED__date"
    ASSET_FIELD_CHANGED_DATE_ADDED = "ASSET_FIELD_CHANGED__date_added"
    ASSET_FIELD_CHANGED_ORIGINAL_FILENAME = "ASSET_FIELD_CHANGED__original_filename"
    ASSET_FIELD_CHANGED_IS_MOVIE = "ASSET_FIELD_CHANGED__is_movie"

    # Asset relationship metadata
    ASSET_ALBUM_MEMBERSHIP_REMOVED = "ASSET_ALBUM_MEMBERSHIP_REMOVED"
    ASSET_ALBUM_MEMBERSHIP_ADDED = "ASSET_ALBUM_MEMBERSHIP_ADDED"

    ASSET_FOLDER_PATHS_REMOVED = "ASSET_FOLDER_PATHS_REMOVED"
    ASSET_FOLDER_PATHS_ADDED = "ASSET_FOLDER_PATHS_ADDED"
    ASSET_FOLDER_PATHS_CHANGED = "ASSET_FOLDER_PATHS_CHANGED"

    # Album existence and folder relationship
    ALBUM_MISSING_FROM_SNAPSHOT = "ALBUM_MISSING_FROM_SNAPSHOT"
    ALBUM_NEW_IN_SNAPSHOT = "ALBUM_NEW_IN_SNAPSHOT"

    ALBUM_FOLDER_PATHS_REMOVED = "ALBUM_FOLDER_PATHS_REMOVED"
    ALBUM_FOLDER_PATHS_ADDED = "ALBUM_FOLDER_PATHS_ADDED"
    ALBUM_FOLDER_PATHS_CHANGED = "ALBUM_FOLDER_PATHS_CHANGED"

    ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS = "ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS"

    # Folder path existence
    FOLDER_PATH_MISSING_FROM_SNAPSHOT = "FOLDER_PATH_MISSING_FROM_SNAPSHOT"
    FOLDER_PATH_NEW_IN_SNAPSHOT = "FOLDER_PATH_NEW_IN_SNAPSHOT"


CHANGE_TYPE_DESCRIPTIONS = {
    ChangeType.ASSET_MISSING_FROM_SNAPSHOT:
        "Asset exists in backup but not in snapshot. This is high-priority because it may mean the photo or video itself disappeared after iCloud Photos crashes.",

    ChangeType.ASSET_NEW_IN_SNAPSHOT:
        "Asset exists in snapshot but not in backup. Usually normal because snapshot is later than backup.",

    ChangeType.ASSET_FIELD_CHANGED_DESCRIPTION:
        "Same asset UUID exists in both libraries, but the caption/description changed or disappeared.",

    ChangeType.ASSET_FIELD_CHANGED_KEYWORDS:
        "Same asset UUID exists in both libraries, but keyword metadata changed or disappeared.",

    ChangeType.ASSET_FIELD_CHANGED_FAVORITE:
        "Same asset UUID exists in both libraries, but favorite status changed. This may be real user action or metadata loss.",

    ChangeType.ASSET_FIELD_CHANGED_HIDDEN:
        "Same asset UUID exists in both libraries, but hidden status changed. This may be real user action or metadata difference.",

    ChangeType.ASSET_FIELD_CHANGED_DATE:
        "Same asset UUID exists in both libraries, but asset date changed. This is unusual and should be inspected carefully.",

    ChangeType.ASSET_FIELD_CHANGED_DATE_ADDED:
        "Same asset UUID exists in both libraries, but date_added changed. This may be less important because import/sync timing can differ.",

    ChangeType.ASSET_FIELD_CHANGED_ORIGINAL_FILENAME:
        "Same asset UUID exists in both libraries, but original filename changed. This is unusual and should be inspected carefully.",

    ChangeType.ASSET_FIELD_CHANGED_IS_MOVIE:
        "Same asset UUID exists in both libraries, but is_movie changed. This is highly unusual.",

    ChangeType.ASSET_ALBUM_MEMBERSHIP_REMOVED:
        "Asset still exists in snapshot, but one or more album memberships from backup are missing.",

    ChangeType.ASSET_ALBUM_MEMBERSHIP_ADDED:
        "Asset has album memberships in snapshot that did not exist in backup. Often normal because snapshot is later.",

    ChangeType.ASSET_FOLDER_PATHS_REMOVED:
        "Asset still exists in snapshot, but backup folder-path relationships are gone in snapshot.",

    ChangeType.ASSET_FOLDER_PATHS_ADDED:
        "Asset has folder-path relationships in snapshot that did not exist in backup. Often normal or caused by later organization.",

    ChangeType.ASSET_FOLDER_PATHS_CHANGED:
        "Asset still exists in both libraries, but folder-path relationships changed.",

    ChangeType.ALBUM_MISSING_FROM_SNAPSHOT:
        "Album title exists in backup but not in snapshot. This may require album reconstruction.",

    ChangeType.ALBUM_NEW_IN_SNAPSHOT:
        "Album title exists in snapshot but not in backup. Usually normal because snapshot is later.",

    ChangeType.ALBUM_FOLDER_PATHS_REMOVED:
        "Album still exists in snapshot, but it is no longer inside the folder path recorded in backup.",

    ChangeType.ALBUM_FOLDER_PATHS_ADDED:
        "Album gained folder-path relationships in snapshot that did not exist in backup.",

    ChangeType.ALBUM_FOLDER_PATHS_CHANGED:
        "Album still exists in both libraries, but its folder path changed.",

    ChangeType.ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS:
        "Album title is duplicated or ambiguous in at least one inventory, so title-based comparison is not reliable for this album.",

    ChangeType.FOLDER_PATH_MISSING_FROM_SNAPSHOT:
        "Folder path exists in backup but not in snapshot. Albums or assets may still exist elsewhere.",

    ChangeType.FOLDER_PATH_NEW_IN_SNAPSHOT:
        "Folder path exists in snapshot but not in backup. Usually normal if the folder was created later.",
}


FIELD_CHANGE_TYPES = {
    "description": ChangeType.ASSET_FIELD_CHANGED_DESCRIPTION,
    "keywords": ChangeType.ASSET_FIELD_CHANGED_KEYWORDS,
    "favorite": ChangeType.ASSET_FIELD_CHANGED_FAVORITE,
    "hidden": ChangeType.ASSET_FIELD_CHANGED_HIDDEN,
    "date": ChangeType.ASSET_FIELD_CHANGED_DATE,
    "date_added": ChangeType.ASSET_FIELD_CHANGED_DATE_ADDED,
    "original_filename": ChangeType.ASSET_FIELD_CHANGED_ORIGINAL_FILENAME,
    "is_movie": ChangeType.ASSET_FIELD_CHANGED_IS_MOVIE,
}

## uuid among different photos libraries is different for the same asset

def make_asset_index(inventory):
    # Build photo_library_asset_unique_id -> Asset object.
    #
    # Photos UUID is local to one Photos Library database.
    # It cannot be used to match the same asset across different
    # Photos Library snapshots or backups.
    index = {}

    for asset in inventory["assets"]:
        unique_id = asset.get("photo_library_asset_unique_id")

        if unique_id is None:
            raise RuntimeError(
                "Asset is missing photo_library_asset_unique_id. "
                "Run fill_photo_library_asset_unique_ids() first."
            )

        if unique_id in index:
            raise RuntimeError(
                "Duplicate photo_library_asset_unique_id found. "
                "Do not run comparison until the unique ID scheme is strengthened."
            )

        index[unique_id] = asset

    return index


def make_album_title_index(inventory):
    # Build album_title -> list[Album object].
    # Album title may not be globally unique, so keep a list.
    index = {}

    for album in inventory["albums"].values():
        title = album["title"] or ""

        if title not in index:
            index[title] = []

        index[title].append(album)

    return index


def make_folder_path_index(inventory):
    # Build folder_path -> list[Folder object].
    # Folder path may theoretically collide, so keep a list.
    index = {}

    for folder in inventory["folders"].values():
        path = folder["path"] or ""

        if path not in index:
            index[path] = []

        index[path].append(folder)

    return index


def asset_display_name(asset):
    # Prefer original filename for human reading.
    if asset is None:
        return None

    return asset["original_filename"] or asset["filename"] or asset["uuid"]


def album_folder_paths(album):
    # Return sorted folder paths for one Album object.
    return sorted(
        folder["path"]
        for folder in album["folders"].values()
    )


def asset_album_titles(asset):
    # Return sorted album titles for one Asset object.
    return sorted(
        album["title"] or ""
        for album in asset["albums"].values()
    )


def asset_folder_paths(asset):
    # Return sorted folder paths for one Asset object.
    return sorted(
        folder["path"] or ""
        for folder in asset["folders"].values()
    )


def add_diff(
    diff_records,
    change_type,
    scope,
    backup_object,
    snapshot_object,
    backup_value,
    snapshot_value,
    changed_field=None,
    note=None,
):
    # Add one normalized comparison record.
    if isinstance(change_type, ChangeType):
        change_type_value = change_type.value
        change_type_description = CHANGE_TYPE_DESCRIPTIONS.get(change_type)
    else:
        raise TypeError(f"change_type must be ChangeType, got: {change_type}")

    record = {
        "change_type": change_type_value,
        "change_type_description": change_type_description,
        "scope": scope,

        "changed_field": changed_field,

        "backup_object": backup_object,
        "snapshot_object": snapshot_object,

        "backup_value": backup_value,
        "snapshot_value": snapshot_value,

        "note": note,
    }

    diff_records.append(record)


def compare_asset_existence(inventory_backup, inventory_snapshot, diff_records):
    # Compare asset UUID existence.
    backup_assets = make_asset_index(inventory_backup)
    snapshot_assets = make_asset_index(inventory_snapshot)

    backup_uuids = set(backup_assets)
    snapshot_uuids = set(snapshot_assets)

    for asset_uuid in sorted(backup_uuids - snapshot_uuids):
        backup_asset = backup_assets[asset_uuid]

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ASSET_MISSING_FROM_SNAPSHOT,
            scope="asset",
            backup_object=backup_asset,
            snapshot_object=None,
            backup_value=asset_display_name(backup_asset),
            snapshot_value=None,
            note="Asset exists in backup but not in snapshot. This is a high-priority possible iCloud crash data-loss case.",
        )

    for asset_uuid in sorted(snapshot_uuids - backup_uuids):
        snapshot_asset = snapshot_assets[asset_uuid]

        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ASSET_NEW_IN_SNAPSHOT,
            scope="asset",
            backup_object=None,
            snapshot_object=snapshot_asset,
            backup_value=None,
            snapshot_value=asset_display_name(snapshot_asset),
            note="Asset exists in snapshot but not in backup. This is usually normal because the snapshot is later.",
        )


def compare_asset_metadata(inventory_backup, inventory_snapshot, diff_records):
    # Compare metadata for assets with the same UUID.
    backup_assets = make_asset_index(inventory_backup)
    snapshot_assets = make_asset_index(inventory_snapshot)

    common_uuids = sorted(set(backup_assets) & set(snapshot_assets))

    fields_to_compare = [
        "original_filename",
        "is_movie",
        "date",
        "date_added",
        "description",
        "keywords",
        "favorite",
        "hidden",
    ]

    for asset_uuid in common_uuids:
        backup_asset = backup_assets[asset_uuid]
        snapshot_asset = snapshot_assets[asset_uuid]

        for field in fields_to_compare:
            backup_value = backup_asset.get(field)
            snapshot_value = snapshot_asset.get(field)

            if backup_value != snapshot_value:
                add_diff(
                    diff_records=diff_records,
                    change_type=FIELD_CHANGE_TYPES[field],
                    scope="asset",
                    backup_object=backup_asset,
                    snapshot_object=snapshot_asset,
                    backup_value=backup_value,
                    snapshot_value=snapshot_value,
                    changed_field=field,
                    note=f"Same asset UUID but asset field changed: {field}",
                )


def compare_asset_album_membership(inventory_backup, inventory_snapshot, diff_records):
    # Compare album membership by asset UUID and album title.
    backup_assets = make_asset_index(inventory_backup)
    snapshot_assets = make_asset_index(inventory_snapshot)

    common_uuids = sorted(set(backup_assets) & set(snapshot_assets))

    for asset_uuid in common_uuids:
        backup_asset = backup_assets[asset_uuid]
        snapshot_asset = snapshot_assets[asset_uuid]

        backup_titles = set(asset_album_titles(backup_asset))
        snapshot_titles = set(asset_album_titles(snapshot_asset))

        removed_titles = sorted(backup_titles - snapshot_titles)
        added_titles = sorted(snapshot_titles - backup_titles)

        if removed_titles:
            add_diff(
                diff_records=diff_records,
                change_type=ChangeType.ASSET_ALBUM_MEMBERSHIP_REMOVED,
                scope="asset_album_membership",
                backup_object=backup_asset,
                snapshot_object=snapshot_asset,
                backup_value=removed_titles,
                snapshot_value=None,
                note="Asset still exists, but some backup album memberships are missing from snapshot.",
            )

        if added_titles:
            add_diff(
                diff_records=diff_records,
                change_type=ChangeType.ASSET_ALBUM_MEMBERSHIP_ADDED,
                scope="asset_album_membership",
                backup_object=backup_asset,
                snapshot_object=snapshot_asset,
                backup_value=None,
                snapshot_value=added_titles,
                note="Asset has album memberships in snapshot that did not exist in backup. Often normal for later snapshot.",
            )


def compare_asset_folder_paths(inventory_backup, inventory_snapshot, diff_records):
    # Compare folder paths attached to the same asset.
    # These are derived through asset -> album_info -> folder_list.
    backup_assets = make_asset_index(inventory_backup)
    snapshot_assets = make_asset_index(inventory_snapshot)

    common_uuids = sorted(set(backup_assets) & set(snapshot_assets))

    for asset_uuid in common_uuids:
        backup_asset = backup_assets[asset_uuid]
        snapshot_asset = snapshot_assets[asset_uuid]

        backup_paths = set(asset_folder_paths(backup_asset))
        snapshot_paths = set(asset_folder_paths(snapshot_asset))

        if backup_paths == snapshot_paths:
            continue

        if backup_paths and not snapshot_paths:
            change_type = ChangeType.ASSET_FOLDER_PATHS_REMOVED
            note = "Asset still exists, but its folder paths are gone in snapshot."
        elif not backup_paths and snapshot_paths:
            change_type = ChangeType.ASSET_FOLDER_PATHS_ADDED
            note = "Asset has folder paths in snapshot but did not have them in backup."
        else:
            change_type = ChangeType.ASSET_FOLDER_PATHS_CHANGED
            note = "Asset still exists, but folder paths changed."

        add_diff(
            diff_records=diff_records,
            change_type=change_type,
            scope="asset_folder_paths",
            backup_object=backup_asset,
            snapshot_object=snapshot_asset,
            backup_value=sorted(backup_paths),
            snapshot_value=sorted(snapshot_paths),
            note=note,
        )


def compare_album_existence_and_folder_paths(inventory_backup, inventory_snapshot, diff_records):
    # Compare albums by album title.
    # Title is the human-facing identity; UUID may not be stable across libraries.
    backup_albums_by_title = make_album_title_index(inventory_backup)
    snapshot_albums_by_title = make_album_title_index(inventory_snapshot)

    backup_titles = set(backup_albums_by_title)
    snapshot_titles = set(snapshot_albums_by_title)

    for title in sorted(backup_titles - snapshot_titles):
        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ALBUM_MISSING_FROM_SNAPSHOT,
            scope="album",
            backup_object=backup_albums_by_title[title],
            snapshot_object=None,
            backup_value=title,
            snapshot_value=None,
            note="Album title exists in backup but not in snapshot. This may require album reconstruction.",
        )

    for title in sorted(snapshot_titles - backup_titles):
        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.ALBUM_NEW_IN_SNAPSHOT,
            scope="album",
            backup_object=None,
            snapshot_object=snapshot_albums_by_title[title],
            backup_value=None,
            snapshot_value=title,
            note="Album title exists in snapshot but not in backup. Usually normal for later snapshot.",
        )

    for title in sorted(backup_titles & snapshot_titles):
        backup_albums = backup_albums_by_title[title]
        snapshot_albums = snapshot_albums_by_title[title]

        if len(backup_albums) != 1 or len(snapshot_albums) != 1:
            add_diff(
                diff_records=diff_records,
                change_type=ChangeType.ALBUM_TITLE_DUPLICATE_OR_AMBIGUOUS,
                scope="album",
                backup_object=backup_albums,
                snapshot_object=snapshot_albums,
                backup_value=len(backup_albums),
                snapshot_value=len(snapshot_albums),
                note="Album title is not unique in at least one inventory. Folder comparison by title is ambiguous.",
            )
            continue

        backup_album = backup_albums[0]
        snapshot_album = snapshot_albums[0]

        backup_paths = set(album_folder_paths(backup_album))
        snapshot_paths = set(album_folder_paths(snapshot_album))

        if backup_paths == snapshot_paths:
            continue

        if backup_paths and not snapshot_paths:
            change_type = ChangeType.ALBUM_FOLDER_PATHS_REMOVED
            note = "Album still exists, but it is no longer inside any folder path in snapshot."
        elif not backup_paths and snapshot_paths:
            change_type = ChangeType.ALBUM_FOLDER_PATHS_ADDED
            note = "Album gained folder paths in snapshot."
        else:
            change_type = ChangeType.ALBUM_FOLDER_PATHS_CHANGED
            note = "Album still exists, but its folder path changed."

        add_diff(
            diff_records=diff_records,
            change_type=change_type,
            scope="album_folder_paths",
            backup_object=backup_album,
            snapshot_object=snapshot_album,
            backup_value=sorted(backup_paths),
            snapshot_value=sorted(snapshot_paths),
            note=note,
        )


def compare_folder_paths(inventory_backup, inventory_snapshot, diff_records):
    # Compare folder paths by human-readable path.
    backup_folders_by_path = make_folder_path_index(inventory_backup)
    snapshot_folders_by_path = make_folder_path_index(inventory_snapshot)

    backup_paths = set(backup_folders_by_path)
    snapshot_paths = set(snapshot_folders_by_path)

    for path in sorted(backup_paths - snapshot_paths):
        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.FOLDER_PATH_MISSING_FROM_SNAPSHOT,
            scope="folder",
            backup_object=backup_folders_by_path[path],
            snapshot_object=None,
            backup_value=path,
            snapshot_value=None,
            note="Folder path exists in backup but not in snapshot. Albums/assets may still exist elsewhere.",
        )

    for path in sorted(snapshot_paths - backup_paths):
        add_diff(
            diff_records=diff_records,
            change_type=ChangeType.FOLDER_PATH_NEW_IN_SNAPSHOT,
            scope="folder",
            backup_object=None,
            snapshot_object=snapshot_folders_by_path[path],
            backup_value=None,
            snapshot_value=path,
            note="Folder path exists in snapshot but not in backup.",
        )


def compare_inventories(inventory_backup, inventory_snapshot):
    # Run all comparison passes and return normalized diff records.
    diff_records = []

    compare_asset_existence(inventory_backup, inventory_snapshot, diff_records)
    compare_asset_metadata(inventory_backup, inventory_snapshot, diff_records)
    compare_asset_album_membership(inventory_backup, inventory_snapshot, diff_records)
    compare_asset_folder_paths(inventory_backup, inventory_snapshot, diff_records)
    compare_album_existence_and_folder_paths(inventory_backup, inventory_snapshot, diff_records)
    compare_folder_paths(inventory_backup, inventory_snapshot, diff_records)

    return diff_records


def summarize_diff_records(diff_records):
    # Count diff records by change_type.
    summary = {}

    for record in diff_records:
        change_type = record["change_type"]

        if change_type not in summary:
            summary[change_type] = 0

        summary[change_type] += 1

    return dict(sorted(summary.items()))


In [ ]:
# ============================================================
# Test 2: Photo Library asset unique ID preflight
# ============================================================

fill_photo_library_asset_unique_ids(inventory_backup)
fill_photo_library_asset_unique_ids(inventory_snapshot)

backup_unique_id_ok = audit_photo_library_asset_unique_ids(
    inventory_backup,
    "BACKUP Photo Library asset unique ID audit",
)

print()

snapshot_unique_id_ok = audit_photo_library_asset_unique_ids(
    inventory_snapshot,
    "SNAPSHOT Photo Library asset unique ID audit",
)

print()
print("backup_unique_id_ok:", backup_unique_id_ok)
print("snapshot_unique_id_ok:", snapshot_unique_id_ok)
print("ready_for_cross_library_comparison:", backup_unique_id_ok and snapshot_unique_id_ok)

In [ ]:
# ============================================================
# Test 2: Run inventory comparison summary
# ============================================================

if not (backup_unique_id_ok and snapshot_unique_id_ok):
    raise RuntimeError(
        "Photo Library asset unique ID preflight failed. "
        "Do not run cross-library comparison yet."
    )

diff_records = compare_inventories(
    inventory_backup=inventory_backup,
    inventory_snapshot=inventory_snapshot,
)

In [ ]:

# ============================================================
# Test 2: Inspect diff records by change type
# ============================================================

target_change_type = ChangeType.ASSET_MISSING_FROM_SNAPSHOT.value

matched_records = [
    record
    for record in diff_records
    if record["change_type"] == target_change_type
]

print("target change type:", target_change_type)
print("matched record count:", len(matched_records))
print()

for record in matched_records[:20]:
    backup_asset = record["backup_object"]
    snapshot_asset = record["snapshot_object"]

    print("change_type:", record["change_type"])
    print("description:", record["change_type_description"])
    print("changed_field:", record["changed_field"])
    print("backup_value:", record["backup_value"])
    print("snapshot_value:", record["snapshot_value"])

    if backup_asset is not None:
        print("backup uuid:", backup_asset["uuid"])
        print("backup original_filename:", backup_asset["original_filename"])
        print("backup date:", backup_asset["date"])

    if snapshot_asset is not None:
        print("snapshot uuid:", snapshot_asset["uuid"])
        print("snapshot original_filename:", snapshot_asset["original_filename"])
        print("snapshot date:", snapshot_asset["date"])

    print("-" * 80)
